# 07 · Student Candidate — MobileNetV3-Small (pretrained ImageNet)

MobileNetV3-Small fine-tuned from ImageNet weights.
**This is the selected student for knowledge distillation.**

**Justification for selection over MobileNetV2-scratch:**
- Hardware-Aware NAS: designed explicitly for mobile/edge inference latency
- SE (Squeeze-and-Excitation) blocks: channel attention improves accuracy/param ratio
- h-swish activations: more efficient than ReLU6 on hardware with lookup tables
- Pretrained weights: better feature initialization on only 7,000 images
- ~2.5M parameters: suitable for STM32 deployment after INT8 quantization

**Two-phase protocol:**
- Phase 1 (epochs 1–9):  backbone frozen, classifier head only at lr=3e-4
- Phase 2 (epochs 10–25): full unfreeze at lr=1e-4


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


## Standardized hyperparameters

All models use identical settings to ensure fair comparison:

| Parameter | Scratch models | Pretrained models |
|-----------|---------------|-------------------|
| Batch size | 64 | 64 |
| Optimizer | Adam | Adam |
| Weight decay | 1e-4 | 1e-4 |
| Label smoothing | 0.1 | 0.1 |
| Augmentation | standard | standard |
| Scheduler | CosineAnnealingLR | CosineAnnealingLR |
| Patience | 10 | 10 |
| Seeds | [41, 52, 63] | [41, 52, 63] |
| Max epochs | 50 | 30 |
| LR | 1e-3 | 3e-4 → 1e-4 → 3e-5 (3-phase) |

Pretrained models use fewer max epochs because transfer learning converges faster.
The three-phase progressive unfreeze is only applicable to pretrained models.


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")


In [ ]:
# MobileNetV3 uses two-phase (not three) — backbone is lighter, full unfreeze is safe at ep 10
results, best = train_multi_seed(
    model_fn     = MobileNetV3_Pretrained,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63],
    save_dir     = SAVE_DIR,
    name_prefix  = "mobilenetv3_baseline",
    pretrained   = True,
    epochs          = 25,
    lr_phase1       = 3e-4,
    lr_phase2       = 1e-4,
    lr_phase3       = 1e-4,   # phase3 same as phase2 (effectively two-phase)
    phase2_epoch    = 10,
    phase3_epoch    = 999,    # disable phase3 transition
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)


In [ ]:
plot_history(best, title=f"MobileNetV3-Small Pretrained (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nMobileNetV3-Small  |  Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")
print(f"Best: {best['best_acc']*100:.2f}% (seed {best['seed']})")
print(f"Checkpoint: {best['save_path']}")
